# VIX Mean Reversion — EDA

Pull VIX daily OHLC from yfinance and explore its properties as a mean-reversion candidate.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Pull data

In [ ]:
raw = yf.download("^VIX", start="1990-01-02", auto_adjust=False)
raw.columns = raw.columns.get_level_values(0)  # flatten MultiIndex if present
raw.index = pd.to_datetime(raw.index)

df = raw[["Open", "High", "Low", "Close"]].copy()
df.columns = [c.lower() for c in df.columns]

# VIX has no volume/VWAP — it's a computed index, not a traded instrument
# Typical proxy for "midpoint" is (high + low) / 2
df["mid"] = (df["high"] + df["low"]) / 2

print(f"Date range : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Rows       : {len(df):,}")
print(f"Missing    : {df['close'].isna().sum()} nulls in close")
df.tail(5)

## 2. Full history — Open & Close

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["open"], color="steelblue", alpha=0.5, lw=0.8, label="Open")
ax.plot(df.index, df["close"], color="navy", lw=0.9, label="Close")
ax.set_title("VIX — Daily Open & Close (1990–present)", fontsize=13)
ax.set_ylabel("VIX level")
ax.legend()

# Annotate notable spikes
events = {
    "2008 GFC":    "2008-10-24",
    "2020 COVID":  "2020-03-18",
    "2010 Flash":  "2010-05-20",
    "2022 Rates":  "2022-03-07",
}
for label, date in events.items():
    d = pd.Timestamp(date)
    if d in df.index:
        v = df.loc[d, "close"]
        ax.annotate(label, xy=(d, v), xytext=(0, 12), textcoords="offset points",
                    fontsize=7, ha="center", arrowprops=dict(arrowstyle="-", color="gray", lw=0.7))

plt.tight_layout()
plt.show()

## 3. OHLC candlestick — last 12 months

In [ ]:
recent = df.last("365D").copy()

fig, ax = plt.subplots(figsize=(14, 4))

for i, (date, row) in enumerate(recent.iterrows()):
    color = "#2ca02c" if row["close"] >= row["open"] else "#d62728"
    # wick
    ax.plot([date, date], [row["low"], row["high"]], color=color, lw=0.5, alpha=0.7)
    # body
    ax.plot([date, date], [row["open"], row["close"]], color=color, lw=2, solid_capstyle="butt")

ax.set_title("VIX — OHLC Candlestick (last 12 months)", fontsize=13)
ax.set_ylabel("VIX level")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Fair value lines — SMA overlays

In [ ]:
windows = [63, 126, 252]  # ~3m, ~6m, ~1y trading days
colors  = ["#e377c2", "#ff7f0e", "#1f77b4"]

for w, c in zip(windows, colors):
    df[f"sma_{w}"] = df["close"].rolling(w, min_periods=w).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["close"], color="navy", lw=0.8, alpha=0.6, label="Close")
for w, c in zip(windows, colors):
    ax.plot(df.index, df[f"sma_{w}"], color=c, lw=1.3, label=f"{w}d SMA (~{'3m' if w==63 else '6m' if w==126 else '1y'})")

ax.set_title("VIX — Close vs. Rolling SMA Fair Value Lines", fontsize=13)
ax.set_ylabel("VIX level")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Deviation from fair value (252d SMA)

In [ ]:
fv = df["sma_252"].dropna()
dev = ((df["close"] - df["sma_252"]) / df["sma_252"] * 100).dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df.index, df["close"], color="navy", lw=0.8, label="Close")
axes[0].plot(fv.index, fv, color="#ff7f0e", lw=1.5, label="252d SMA")
axes[0].set_ylabel("VIX")
axes[0].legend()
axes[0].set_title("VIX vs 252d SMA fair value", fontsize=12)

axes[1].axhline(0, color="black", lw=0.8)
axes[1].axhline(10, color="green", lw=1, ls="--", label="+10% (short entry)")
axes[1].axhline(-10, color="red", lw=1, ls="--", label="-10% (long entry)")
axes[1].fill_between(dev.index, dev, 0, where=(dev > 10), color="green", alpha=0.2)
axes[1].fill_between(dev.index, dev, 0, where=(dev < -10), color="red", alpha=0.2)
axes[1].plot(dev.index, dev, color="purple", lw=0.7, alpha=0.8)
axes[1].set_ylabel("Deviation (%)")
axes[1].legend(fontsize=9)
axes[1].set_title("% deviation from fair value — entry zones shaded", fontsize=12)

plt.tight_layout()
plt.show()

print(f"Days below -10% (long signal) : {(dev < -10).sum():,} ({(dev < -10).mean():.1%} of days)")
print(f"Days above +10% (short signal): {(dev >  10).sum():,} ({(dev >  10).mean():.1%} of days)")
print(f"Mean deviation                : {dev.mean():.1f}%")
print(f"Std deviation                 : {dev.std():.1f}%")

## 6. Distribution of VIX close values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["close"].dropna(), bins=60, color="navy", alpha=0.7, edgecolor="white", linewidth=0.3)
axes[0].set_xlabel("VIX close")
axes[0].set_ylabel("Days")
axes[0].set_title("Distribution of VIX close (1990–present)")
for pct, label in [(25, "Q1"), (50, "Median"), (75, "Q3")]:
    v = np.percentile(df["close"].dropna(), pct)
    axes[0].axvline(v, color="orange", lw=1, ls="--")
    axes[0].text(v + 0.5, axes[0].get_ylim()[1] * 0.9, f"{label}={v:.0f}", fontsize=8, color="orange")

axes[1].hist(dev.dropna(), bins=60, color="purple", alpha=0.7, edgecolor="white", linewidth=0.3)
axes[1].set_xlabel("% deviation from 252d SMA")
axes[1].set_ylabel("Days")
axes[1].set_title("Distribution of deviation from fair value")
axes[1].axvline(0, color="black", lw=1)
axes[1].axvline(10, color="green", lw=1, ls="--", label="+10%")
axes[1].axvline(-10, color="red", lw=1, ls="--", label="-10%")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(df["close"].describe().round(1))

## 7. Mean-reversion check — ADF test

In [ ]:
from statsmodels.tsa.stattools import adfuller

series = df["close"].dropna()
result = adfuller(series, autolag="AIC")

print("ADF Test — VIX Close")
print(f"  Test statistic : {result[0]:.4f}")
print(f"  p-value        : {result[1]:.6f}")
print(f"  Critical values: { {k: f'{v:.3f}' for k, v in result[4].items()} }")
print()
if result[1] < 0.05:
    print("✅ p < 0.05 — reject unit root — VIX is stationary (mean-reverting). Strategy foundation is valid.")
else:
    print("❌ p >= 0.05 — cannot reject unit root — VIX may not be mean-reverting. Interpret strategy with caution.")

## 8. Open-to-close intraday range

In [ ]:
df["oc_range"] = (df["close"] - df["open"])
df["hl_range"] = df["high"] - df["low"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["oc_range"].dropna(), bins=60, color="steelblue", alpha=0.8, edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="black", lw=1)
axes[0].set_title("Open-to-Close daily change")
axes[0].set_xlabel("Close − Open (VIX points)")

axes[1].hist(df["hl_range"].dropna(), bins=60, color="coral", alpha=0.8, edgecolor="white", linewidth=0.3)
axes[1].set_title("Daily High−Low range")
axes[1].set_xlabel("High − Low (VIX points)")

plt.tight_layout()
plt.show()

print(f"Avg open→close move : {df['oc_range'].mean():.3f} pts  |  std {df['oc_range'].std():.2f}")
print(f"Avg high−low range  : {df['hl_range'].mean():.3f} pts  |  std {df['hl_range'].std():.2f}")
print(f"% days close > open : {(df['oc_range'] > 0).mean():.1%}")